<a href="https://colab.research.google.com/github/NguyenNguyen1504/GROUP-5-BPFE/blob/main/GROUP_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Setup path and read input**



In [24]:
from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')

base_dataset_path = '/content/drive/MyDrive/EPL_data/results.csv'

if os.path.exists(base_dataset_path):
    df_results = pd.read_csv(base_dataset_path, encoding="latin1") # Base dataset
    print("Read data successfully.")
    display(df_results.head())
else:
    print("Error: file not found.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Read data successfully.


,Season,DateTime,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
0,1993-94,1993-08-14T00:00:00Z,Arsenal,Coventry,0,3,A,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1993-94,1993-08-14T00:00:00Z,Aston Villa,QPR,4,1,H,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1993-94,1993-08-14T00:00:00Z,Chelsea,Blackburn,1,2,A,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1993-94,1993-08-14T00:00:00Z,Liverpool,Sheffield Weds,2,0,H,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1993-94,1993-08-14T00:00:00Z,Man City,Leeds,1,1,D,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### **Clean data**

In [25]:
columns_to_keep = ["Season", "DateTime", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
# 93-94 and 94-95 seasons had 22 teams. 21-22 season is missing many matches
rows_to_drop = ["1993-94", "1994-95", "2021-22"]
df_results = df_results[columns_to_keep].copy()
df_results = df_results[~df_results['Season'].isin(rows_to_drop)].copy()

### **Create dictionaries saving data of clubs' home stadiums capacities**

**Source: Wikipedia, NBC Sports** (https://www.nbcsports.com/soccer/news/list-of-premier-league-stadiums-every-clubs-current-and-former-ground-from-the-pl-era)

In [26]:
# Teams with changed stadiums
changes = {
    'Arsenal':       [('2006-07', 60704), ('1993-94', 38419)],
    'Man City':      [('2003-04', 53400), ('1993-94', 35150)],
    'Southampton':   [('2001-02', 32384), ('1993-94', 15200)],
    'Leicester':     [('2002-03', 32261), ('1993-94', 22000)],
    'Middlesbrough': [('1995-96', 34742), ('1993-94', 26667)],
    'West Ham':      [('2016-17', 60000), ('1993-94', 35016)],
    'Tottenham':     [('2019-20', 62850), ('1993-94', 36284)],
    'Bolton':        [('1997-98', 28723), ('1993-94', 22000)],
    'Sunderland':    [('1997-98', 49000), ('1993-94', 22000)]
}

# Teams with unchanged stadiums
fixed = {
    'Man United':       74197,
    'Liverpool':        54074,
    'Newcastle':        52258,
    'Aston Villa':      42785,
    'Chelsea':          40173,
    'Everton':          39414,
    'Sheffield Weds':   39812,
    'Leeds':            37890,
    'Blackburn':        31367,
    'Wolves':           32050,
    'Sheffield United': 32050,
    'Brighton':         31876,
    'Ipswich':          29813,
    'Birmingham':       29409,
    'Stoke':            27902,
    'Norwich':          27359,
    'Charlton':         27111,
    'West Brom':        26850,
    'Wimbledon':        26309,
    'Fulham':           25700,
    'Crystal Palace':   25486,
    'Hull':             25400,
    'Wigan':            25138,
    'Bradford':         25136,
    'Huddersfield':     24500,
    'Reading':          24161,
    'Barnsley':         23009,
    'Watford':          22200,
    'Burnley':          21944,
    'Swansea':          21088,
    'Portsmouth':       20700,
    'QPR':              18360,
    'Brentford':        17250,
    'Blackpool':        16220,
    'Swindon':          15728,
    'Oldham':           13512,
    'Bournemouth':      11307,
    'Derby':            33597,
    'Cardiff':          33280,
    'Coventry':         32609,
    "Nott'm Forest":    30404
}

### **Create a column for home stadium capacity in the big dataset**

In [32]:
def get_capacity(team, season):

    if team in changes:
        for from_season, cap in changes[team]:
            if season >= from_season:
                return cap

    if team in fixed:
        return fixed[team]

    print(f"WARNING: No capacity data for team '{team}'")
    return None

df_results['HomeTeam_StadiumCapacity'] = df_results.apply(lambda row: get_capacity(row['HomeTeam'],row['Season']), axis=1)
df_results = df_results.reset_index(drop=True)

display(df_results.head())

,Season,DateTime,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HomeTeam_StadiumCapacity
0,1995-96,1995-08-19T00:00:00Z,Aston Villa,Man United,3,1,H,42785
1,1995-96,1995-08-19T00:00:00Z,Blackburn,QPR,1,0,H,31367
2,1995-96,1995-08-19T00:00:00Z,Chelsea,Everton,0,0,D,40173
3,1995-96,1995-08-19T00:00:00Z,Liverpool,Sheffield Weds,1,0,H,54074
4,1995-96,1995-08-19T00:00:00Z,Man City,Tottenham,1,1,D,35150


In [41]:
df_results["CapacityGroup"] = pd.cut(
    df_results["HomeTeam_StadiumCapacity"],
    bins = 3,
    labels = ['Small', 'Medium', 'Large']
)

In [38]:
df_home = df_results.groupby(["Season","HomeTeam"]).agg(
    HomeWinRate = ("FTR", lambda col: (col == "H").mean()),
    StadiumCapacity = ("HomeTeam_StadiumCapacity", "first")
).reset_index()
display(df_home.head())

,Season,HomeTeam,HomeWinRate,StadiumCapacity
0,1995-96,Arsenal,0.526316,38419
1,1995-96,Aston Villa,0.578947,42785
2,1995-96,Blackburn,0.736842,31367
3,1995-96,Bolton,0.263158,22000
4,1995-96,Chelsea,0.368421,40173
